Purpose of this notebook is to compare solver-configuration combinations from RectPack and PackingSolver packages, the metric used will be:

| Priority | Metric                  | Meaning                                       |
| -------- | ----------------------- | --------------------------------------------- |
| 1        | sheets_used             | Primary material-use objective                |
| 2        | runtime_seconds         | Practical time cost                           |
| 3        | utilisation             | How densely the used sheets are filled        |
| 4        | sheet_gap_to_area_bound | Distance from a simple area-based lower bound |

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # If notebook is inside notebooks/
RESULTS_DIR = PROJECT_ROOT / "results"

In [ ]:
import pandas as pd

rectpack_results = pd.read_csv(RESULTS_DIR / "rectpack/rectpack_benchmark_raw.csv")
packingsolver_results = pd.read_csv(RESULTS_DIR / "packingsolver/packingsolver_benchmark_raw.csv")

In [6]:
display(rectpack_results.head())


,solver,config_id,algorithm,sort,rotation,bin_selection,sheet_width,sheet_height,available_sheets,runtime_seconds,...,part_fraction,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound,dataset_id,seed,profile,n_parts,repeat
0,rectpack,MaxRectsBaf__long_side__BBF,MaxRectsBaf,long_side,True,BBF,2500,1250,35,0.003368,...,1.0,1.0,0.768552,3,0,rect_v1_strips_sheet2500x1250_n35_seed2,2,strips,35,0
1,rectpack,MaxRectsBssf__area__BBF,MaxRectsBssf,area,False,BBF,2500,1250,100,0.025233,...,1.0,1.0,0.551820,20,16,rect_v1_large_and_small_sheet2500x1250_n100_seed3,3,large_and_small,100,0
2,rectpack,SkylineBl__long_side__BBF,SkylineBl,long_side,True,BBF,2500,1250,100,0.037747,...,1.0,1.0,0.748586,10,3,rect_v1_mixed_sheet2500x1250_n100_seed3,3,mixed,100,0
3,rectpack,MaxRectsBssf__input_order__BBF,MaxRectsBssf,input_order,True,BBF,2500,1250,100,0.003677,...,1.0,1.0,0.808841,10,2,rect_v1_mixed_sheet2500x1250_n100_seed4,4,mixed,100,0
4,rectpack,MaxRectsBaf__long_side__BBF,MaxRectsBaf,long_side,True,BBF,2500,1250,35,0.001230,...,1.0,1.0,0.797313,3,0,rect_v1_strips_sheet2500x1250_n35_seed0,0,strips,35,0


In [4]:
display(packingsolver_results.head())

,solver,config_id,objective,rotation,time_limit_seconds,sheet_width,sheet_height,available_sheets,runtime_seconds,layout_valid,...,part_fraction,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound,dataset_id,seed,profile,n_parts,repeat
0,packingsolver,BinPacking__time30s,BinPacking,True,30,2500,1250,100,10.688380,True,...,1.0,1.0,0.522245,20,18,rect_v1_large_and_small_sheet2500x1250_n100_seed1,1,large_and_small,100,0
1,packingsolver,BinPacking__time30s,BinPacking,False,30,2500,1250,35,0.597895,True,...,1.0,1.0,0.799891,3,0,rect_v1_strips_sheet2500x1250_n35_seed3,3,strips,35,0
2,packingsolver,BinPacking__time30s,BinPacking,True,30,2500,1250,100,30.229841,True,...,1.0,1.0,0.884692,10,1,rect_v1_mixed_sheet2500x1250_n100_seed3,3,mixed,100,0
3,packingsolver,BinPacking__time30s,BinPacking,True,30,2500,1250,35,0.497239,True,...,1.0,1.0,0.797313,3,0,rect_v1_strips_sheet2500x1250_n35_seed0,0,strips,35,0
4,packingsolver,BinPacking__time30s,BinPacking,False,30,2500,1250,100,31.916142,True,...,1.0,1.0,0.844404,7,1,rect_v1_strips_sheet2500x1250_n100_seed1,1,strips,100,0


create a shared method label, combine only the columns needed for comparison and check that the datasets align:

In [7]:
rectpack_results["method"] = (
    "rectpack__"
    + rectpack_results["algorithm"]
    + "__"
    + rectpack_results["sort"]
    + "__BBF"
)

packingsolver_results["method"] = (
    "packingsolver__BinPacking__"
    + packingsolver_results["time_limit_seconds"].astype(str)
    + "s"
)

In [8]:
shared_columns = [
    "dataset_id",
    "seed",
    "profile",
    "n_parts",
    "sheet_width",
    "sheet_height",
    "rotation",
    "solver",
    "method",
    "complete",
    "parts_packed",
    "parts_total",
    "sheets_used",
    "utilisation",
    "runtime_seconds",
]

comparison_results = pd.concat(
    [
        rectpack_results[shared_columns],
        packingsolver_results[shared_columns],
    ],
    ignore_index=True,
)

In [9]:
rectpack_ids = set(rectpack_results["dataset_id"])
packingsolver_ids = set(packingsolver_results["dataset_id"])

print("Only in RectPack:", rectpack_ids - packingsolver_ids)
print("Only in PackingSolver:", packingsolver_ids - rectpack_ids)

Only in RectPack: set()
Only in PackingSolver: set()


Rank each method - for each dataset and rotation setting, identify the best sheet count and then summarise each method:

In [10]:
comparison_results["best_sheets_used"] = (
    comparison_results
    .groupby(["dataset_id", "rotation"])["sheets_used"]
    .transform("min")
)

comparison_results["is_best_sheets"] = (
    comparison_results["sheets_used"]
    == comparison_results["best_sheets_used"]
)

In [11]:
method_summary = (
    comparison_results
    .groupby(["rotation", "solver", "method"])
    .agg(
        runs=("dataset_id", "size"),
        complete_runs=("complete", "sum"),
        mean_sheets_used=("sheets_used", "mean"),
        median_sheets_used=("sheets_used", "median"),
        best_or_tied_runs=("is_best_sheets", "sum"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        median_runtime_seconds=("runtime_seconds", "median"),
        mean_utilisation=("utilisation", "mean"),
    )
    .reset_index()
    .sort_values(
        ["rotation", "mean_sheets_used", "median_runtime_seconds"],
        ascending=[True, True, True],
    )
)

display(method_summary)

,rotation,solver,method,runs,complete_runs,mean_sheets_used,median_sheets_used,best_or_tied_runs,mean_runtime_seconds,median_runtime_seconds,mean_utilisation
0,False,packingsolver,packingsolver__BinPacking__30s,30,30,13.466667,9.5,30,10.684481,1.208018,0.726497
11,False,rectpack,rectpack__MaxRectsBssf__long_side__BBF,30,30,13.800000,10.0,21,0.003194,0.002172,0.686840
3,False,rectpack,rectpack__GuillotineBafSas__long_side__BBF,30,30,13.866667,10.0,18,0.002508,0.001635,0.676762
7,False,rectpack,rectpack__MaxRectsBaf__long_side__BBF,30,30,13.866667,10.0,19,0.003623,0.003114,0.680235
9,False,rectpack,rectpack__MaxRectsBssf__area__BBF,30,30,13.900000,10.0,19,0.004225,0.003109,0.666762
5,False,rectpack,rectpack__MaxRectsBaf__area__BBF,30,30,13.900000,10.0,19,0.003696,0.003505,0.663741
12,False,rectpack,rectpack__MaxRectsBssf__short_side__BBF,30,30,13.933333,10.0,17,0.003369,0.002551,0.664666
8,False,rectpack,rectpack__MaxRectsBaf__short_side__BBF,30,30,13.933333,10.0,17,0.003104,0.002766,0.661818
16,False,rectpack,rectpack__SkylineBl__short_side__BBF,30,30,14.333333,11.0,11,0.010331,0.007131,0.635546
15,False,rectpack,rectpack__SkylineBl__long_side__BBF,30,30,14.533333,10.5,5,0.011256,0.008712,0.626885


Now compare per dataset, not only on average, then count wins:

This answers a question of which method most often produces the fewest sheets?

In [12]:
best_by_dataset = (
    comparison_results
    .loc[comparison_results["complete"]]
    .sort_values(
        ["dataset_id", "rotation", "sheets_used", "runtime_seconds"],
        ascending=[True, True, True, True],
    )
    .groupby(["dataset_id", "rotation"], as_index=False)
    .first()
)

display(
    best_by_dataset[
        [
            "dataset_id",
            "rotation",
            "solver",
            "method",
            "sheets_used",
            "utilisation",
            "runtime_seconds",
        ]
    ]
)

,dataset_id,rotation,solver,method,sheets_used,utilisation,runtime_seconds
0,rect_v1_large_and_small_sheet2500x1250_n100_seed0,False,rectpack,rectpack__MaxRectsBssf__input_order__BBF,44,0.523261,0.005657
1,rect_v1_large_and_small_sheet2500x1250_n100_seed0,True,rectpack,rectpack__GuillotineBafSas__long_side__BBF,42,0.548178,0.007388
2,rect_v1_large_and_small_sheet2500x1250_n100_seed1,False,rectpack,rectpack__MaxRectsBaf__input_order__BBF,39,0.508854,0.005818
3,rect_v1_large_and_small_sheet2500x1250_n100_seed1,True,rectpack,rectpack__MaxRectsBaf__area__BBF,38,0.522245,0.007992
4,rect_v1_large_and_small_sheet2500x1250_n100_seed2,False,rectpack,rectpack__GuillotineBafSas__long_side__BBF,42,0.509779,0.006423
5,rect_v1_large_and_small_sheet2500x1250_n100_seed2,True,rectpack,rectpack__MaxRectsBaf__short_side__BBF,39,0.548993,0.008091
6,rect_v1_large_and_small_sheet2500x1250_n100_seed3,False,rectpack,rectpack__MaxRectsBaf__short_side__BBF,36,0.551820,0.006849
7,rect_v1_large_and_small_sheet2500x1250_n100_seed3,True,rectpack,rectpack__MaxRectsBaf__short_side__BBF,34,0.584280,0.007275
8,rect_v1_large_and_small_sheet2500x1250_n100_seed4,False,rectpack,rectpack__GuillotineBafSas__long_side__BBF,40,0.469561,0.006231
9,rect_v1_large_and_small_sheet2500x1250_n100_seed4,True,rectpack,rectpack__MaxRectsBssf__area__BBF,39,0.481601,0.008192


In [13]:
display(
    best_by_dataset
    .groupby(["rotation", "solver", "method"])
    .size()
    .rename("best_dataset_runs")
    .reset_index()
    .sort_values(
        ["rotation", "best_dataset_runs"],
        ascending=[True, False],
    )
)

,rotation,solver,method,best_dataset_runs
2,False,rectpack,rectpack__GuillotineBafSas__long_side__BBF,14
0,False,packingsolver,packingsolver__BinPacking__30s,6
5,False,rectpack,rectpack__MaxRectsBaf__short_side__BBF,2
7,False,rectpack,rectpack__MaxRectsBssf__input_order__BBF,2
8,False,rectpack,rectpack__MaxRectsBssf__long_side__BBF,2
1,False,rectpack,rectpack__GuillotineBafSas__area__BBF,1
3,False,rectpack,rectpack__MaxRectsBaf__input_order__BBF,1
4,False,rectpack,rectpack__MaxRectsBaf__long_side__BBF,1
6,False,rectpack,rectpack__MaxRectsBssf__area__BBF,1
12,True,rectpack,rectpack__GuillotineBafSas__long_side__BBF,13
